# DeepLab v1（PASCAL VOC 2007を用いたセマンティックセグメンテーション）

---
## 目的
セマンティックセグメンテーションの代表的な手法の一つであるDeepLab v1を，PASCAL VOC 2007データセットを用いて実装します．DeepLab v1は，通常のCNNが持つプーリングによる大きな解像度低下（VGG16では入力の1/32まで縮小される）を，穴あき畳み込み（Atrous Convolution / Dilated Convolution）に置き換えることで抑え，比較的高い解像度（1/8）の特徴マップを保ったままクラス分類を行う手法です．`fcn.ipynb`で実装したFCNが，浅い層のスコアを加算するスキップ接続と学習可能な転置畳み込みによって段階的に解像度を復元していたのに対し，DeepLab v1はバックボーン自体の解像度をあまり落とさないことで，最後に単純な双線形補間を1回行うだけで済む，より単純な設計になっている点が大きな特徴です．なお，原論文（Chen et al., 2015）で後処理として用いられているFully Connected CRFは，本ノートブックでは実装しません．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCSegmentation
from torchvision.models import vgg16, VGG16_Weights
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassJaccardIndex
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．VOCデータセットは20クラスの物体に対して画素単位のクラスラベルが付与されたデータセットで，学習用（trainval）422枚，評価用（test）210枚の画像から構成されます．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を，VOCデータセットの詳細については`fcn.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCSegmentation`で読み込みます．マスク画像は，各画素の値がクラスID（0が背景，1〜20が物体クラス，255が物体の境界など評価から除外する「無視領域」）になっている画像として与えられます．

In [ ]:
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
n_class = len(VOC_CLASSES)  # 21（背景を含む）
IGNORE_INDEX = 255  # 物体境界などの無視領域
IMG_SIZE = 256

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


class VOCSegmentationDataset(torch.utils.data.Dataset):
    """VOCSegmentationをラップし，画像とマスクを同じサイズにリサイズしたTensorとして返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCSegmentation(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, mask = self.voc[idx]
        image = self.img_transform(image)
        # マスクは値がそのままクラスIDのため，補間で値が混ざらないよう最近傍補間でリサイズする
        mask = TF.resize(mask, [self.img_size, self.img_size], interpolation=TF.InterpolationMode.NEAREST)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        return image, mask


batch_size = 8
train_data = VOCSegmentationDataset('trainval')
test_data = VOCSegmentationDataset('test')
print('train:', len(train_data), 'test:', len(test_data))

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

image, mask = train_data[0]
print('image:', image.shape, 'mask:', mask.shape, 'mask unique:', torch.unique(mask))

## VOCカラーパレットと可視化用関数
クラスIDの2次元配列（マスク）をそのまま画像として表示しても分かりにくいため，各クラスIDに固有の色を割り当ててカラー画像に変換する関数を用意します（`fcn.ipynb`と同じ実装です）．

In [ ]:
def voc_colormap(n=n_class):
    """VOCdevkitで標準的に使われる、クラスIDから固有のRGB色を生成する関数"""
    cmap = np.zeros((n, 3), dtype=np.uint8)
    for i in range(n):
        r = g = b = 0
        c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j)
            g |= ((c >> 1) & 1) << (7 - j)
            b |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
        cmap[i] = [r, g, b]
    return cmap


VOC_COLORMAP = voc_colormap()


def decode_segmap(mask):
    """クラスIDのマスク(H, W)を，可視化用のカラー画像(H, W, 3)に変換する"""
    mask = mask.clone()
    mask[mask == IGNORE_INDEX] = 0  # 可視化のため，無視領域は背景色として扱う
    return VOC_COLORMAP[mask.cpu().numpy()]

## DeepLab v1
DeepLab v1（Chen et al., 2015）は，画像分類用CNNのプーリングによる解像度低下を，**Atrous Convolution（Dilated Convolution，穴あき畳み込み）** に置き換えることで抑制し，比較的高い解像度の特徴マップのまま最終層まで到達できるようにしたセグメンテーションモデルです．

Atrous Convolutionは，畳み込みカーネルの各要素の間隔（dilation rate）を広げてサンプリングする畳み込みです．3x3のカーネルにdilation rate `r`を適用すると，パラメータ数・計算量を増やさずに，通常の`(3-1)*r+1`四方のカーネルと同じ受容野を得られます．この性質を利用し，特徴マップをダウンサンプリングせずに（＝プーリングのstrideを1にしたまま）畳み込みの受容野だけを広げることで，解像度と受容野の広さを両立させます．

### バックボーン（dilated VGG16）
事前学習済みVGG16の`features`を，FCNと同様にプーリングの区切りで3つのブロック（`pool3`, `conv4`, `conv5`）に分割します．`pool3`まで（conv1〜conv3）は通常のVGG16のまま変更しません（出力stride 8）．

- `conv4`ブロック：末尾のMaxPoolのみ`stride=1`に変更し，これ以上の解像度低下を止めます（畳み込み自体は通常のまま）．
- `conv5`ブロック：`conv4`のMaxPoolを止めた影響で，本来はconv5がstride16の特徴マップに畳み込むはずが，stride8のままの特徴マップに畳み込むことになります．受容野の縮小を補うため，3つの3x3畳み込みを`dilation=2`のAtrous Convolutionに変更します（カーネルサイズ・事前学習済みの重みはそのまま利用でき，dilationとpaddingの設定だけを変更すれば良い点がポイントです）．末尾のMaxPoolも`stride=1`に変更します．

この結果，バックボーン全体を通しても出力stride 8が保たれます（通常のVGG16ではstride 32）．

全結合層（fc6, fc7）はFCNと同様に畳み込みに変換しますが，pool4・pool5のstrideをともに1にした影響（合計4倍の縮小を打ち消した影響）を補うため，fc6の7x7畳み込みには`dilation=4`を設定します（原論文の"hole algorithm"）．

In [ ]:
class DeepLabV1(nn.Module):
    """VGG16をバックボーンとしたDeepLab v1（LargeFOV版）"""
    def __init__(self, n_class=n_class):
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
        self.pool3 = vgg[:17]  # conv1_1 ... pool3, 出力stride 8,  channels 256（変更なし）

        conv4 = list(vgg[17:24])
        conv4[-1] = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)  # pool4：strideのみ1に変更（畳み込みは通常のまま）
        self.conv4 = nn.Sequential(*conv4)  # conv4_1 ... pool4, 出力strideは8のまま（元は16）

        conv5 = list(vgg[24:31])
        for layer in conv5:
            if isinstance(layer, nn.Conv2d):
                # カーネルサイズ・重みは変えず，dilationとpaddingだけ変更してAtrous Convolution化する
                layer.dilation, layer.padding = (2, 2), (2, 2)
        conv5[-1] = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)  # pool5もstrideを1に変更
        self.conv5 = nn.Sequential(*conv5)  # conv5_1 ... pool5, 出力strideは8のまま（元は32）

        # 全結合層を畳み込みに変換（FCNと同様）．pool4・pool5のstrideをなくした分（4倍）を補うため，
        # fc6の7x7畳み込みはdilation=4のhole algorithmを適用する
        self.fc = nn.Sequential(
            nn.Conv2d(512, 4096, kernel_size=7, padding=12, dilation=4), nn.ReLU(inplace=True), nn.Dropout2d(0.5),
            nn.Conv2d(4096, 4096, kernel_size=1), nn.ReLU(inplace=True), nn.Dropout2d(0.5),
        )
        self.classifier = nn.Conv2d(4096, n_class, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[-2:]
        h = self.pool3(x)
        h = self.conv4(h)
        h = self.conv5(h)
        h = self.fc(h)
        h = self.classifier(h)
        # FCNのようなスキップ接続・転置畳み込みは使わず，単純な双線形補間だけで入力解像度に戻す
        h = F.interpolate(h, size=input_size, mode='bilinear', align_corners=False)
        return h

### アップサンプリング：FCNとの違い
`fcn.ipynb`のFCN-8sは，pool5出力（stride32）を`ConvTranspose2d`でアップサンプリングしつつ，pool4・pool3のスコアを段階的に加算するスキップ接続によって高解像度な情報を補っていました．

これに対しDeepLab v1は，バックボーン自体がstride8までしか解像度を落とさないため，スキップ接続や学習可能な転置畳み込みを使わず，分類層の出力に対して`F.interpolate(mode='bilinear')`による**単純な双線形補間を1回行うだけ**で入力解像度まで復元します．学習パラメータを持たない補間のみでアップサンプリングを行う点が，FCNとの最大の違いです．

In [ ]:
# 出力ストライドの確認：dilated化したバックボーンを通しても解像度が1/8までしか落ちないことを確認する
_model = DeepLabV1(n_class=n_class).to(device)
with torch.no_grad():
    _x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    _h = _model.conv5(_model.conv4(_model.pool3(_x)))
    print('入力:', _x.shape, '-> conv5出力:', _h.shape)  # (1, 512, 32, 32) であればstride 8

    _out = _model(_x)
    print('最終出力:', _out.shape)  # (1, n_class, 256, 256)：入力と同じ解像度に復元されている

del _model

## 損失関数
FCNと同様，画素ごとのクラス分類問題として`nn.CrossEntropyLoss`を用います．無視領域（ラベル255）は`ignore_index`で損失計算から除外します．

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

## ネットワークの作成
DeepLab v1を構築し，最適化手法としてAdamを設定します．バックボーンに事前学習済みの重みを使用しているため，FCNと同様に小さめの学習率を用います．

In [ ]:
model = DeepLabV1(n_class=n_class).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

num_params = sum(p.numel() for p in model.parameters())
print('パラメータ数:', num_params)

## 学習
学習データ（trainval）を用いてDeepLab v1を学習します．VOC2007のセグメンテーション用データは422枚と少ないため，`epoch_num`を多めに設定しています．

In [ ]:
epoch_num = 30

model.train()
for epoch in range(epoch_num):
    sum_loss = 0.0
    count = 0
    for image, mask in train_loader:
        image, mask = image.to(device), mask.to(device)

        optimizer.zero_grad()
        output = model(image)
        loss = criterion(output, mask)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item() * image.size(0)
        count += image.size(0)

    print(f'epoch: {epoch + 1}, mean loss: {sum_loss / count:.4f}')

## テスト（mIoU評価）
評価用データ（test）に対して，クラスごとのIoUの平均であるmIoU（mean IoU）を求めます．mIoUの算出には`torchmetrics.classification.MulticlassJaccardIndex`を使用し，無視領域（ラベル255）は`ignore_index`で評価から除外します．

In [ ]:
metric = MulticlassJaccardIndex(num_classes=n_class, ignore_index=IGNORE_INDEX, average='macro').to(device)

model.eval()
with torch.no_grad():
    for image, mask in test_loader:
        image, mask = image.to(device), mask.to(device)
        output = model(image)
        pred = output.argmax(dim=1)
        metric.update(pred, mask)

print(f'mIoU: {metric.compute().item():.4f}')

## セグメンテーション結果の可視化
評価用データから1枚取り出し，入力画像・正解マスク・予測マスクを並べて表示します．

In [ ]:
model.eval()
image, mask = test_data[0]
with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    pred = output.argmax(dim=1).squeeze(0).cpu()

# 表示用に正規化前の画素値へ戻す
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
image_vis = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_vis); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(decode_segmap(mask)); axes[1].set_title('ground truth'); axes[1].axis('off')
axes[2].imshow(decode_segmap(pred)); axes[2].set_title('prediction'); axes[2].axis('off')
plt.show()

## オリジナルのDeepLab v1との違い

| 項目 | オリジナル論文 (Chen et al., 2015) | 本ノートブック |
| :-- | :-- | :-- |
| バックボーンの初期化 | conv5・fc6・fc7の重みも事前学習済みVGG16の重みをsub-samplingなどで転用して初期化 | conv5は事前学習済みの重みをそのままAtrous Convolutionとして流用，fc6・fc7はランダム初期化（FCNと同様） |
| 後処理 | Fully Connected CRFで物体境界を精緻化 | CRFによる後処理は省略（`argmax`によるクラス決定のみ） |
| マルチスケール入力 | 複数スケールの入力に対する予測を融合 | 単一スケール（256x256）のみ |
| 最適化手法 | SGD + momentum，step方式の学習率減衰 | Adam（lr=1e-4） |
| 学習データ | VOCの拡張データ（SBDなど）を含む大規模データ | VOC2007のtrainvalのみ（422枚） |

## 課題

1. fc6の`dilation`を4ではなく1（通常のVGG16のfc6と同じ）にした場合，出力ストライドや受容野・出力の形状がどう変化するか考え，実際に変更して精度を比較してください．
2. 本ノートブックでは省略したFully Connected CRFを`pydensecrf`などのライブラリを用いて後処理として追加し，mIoUがどのように変化するか確認してください．
3. アップサンプリングを`F.interpolate`ではなく，FCNのような学習可能な転置畳み込みに置き換えて実装し，精度や学習パラメータ数の違いを比較してください．